In [1]:
import torch   
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

with_cuda = torch.cuda.is_available()
if with_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import os
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import random_split, DataLoader, Dataset
from torch.amp import GradScaler
from torch.amp import autocast
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [2]:
def create_one_hot_dico():   
    nom = ['george', 'jackson', 'lucas', 'nicolas', 'theo', 'yweweler']
    filtre = {}
    num_classes = len(nom)  # Nombre total de classes

    for idx, name in enumerate(nom):
        # Créer un vecteur one-hot pour chaque nom
        one_hot = [0] * num_classes
        one_hot[idx] = 1
        filtre[name] = torch.tensor(one_hot, dtype=torch.float32)  # Associer le nom au vecteur one-hot

    return filtre 

one_hot_dico = create_one_hot_dico()
print(one_hot_dico)

def avoironehot(personnes): #montrer ex au prof 
    if isinstance(personnes, (list, tuple)):  # Si c'est un batch (liste de noms)
        return torch.stack([one_hot_dico[p] for p in personnes])
    else:  # Si c'est un seul nom
        return one_hot_dico[personnes]


{'george': tensor([1., 0., 0., 0., 0., 0.]), 'jackson': tensor([0., 1., 0., 0., 0., 0.]), 'lucas': tensor([0., 0., 1., 0., 0., 0.]), 'nicolas': tensor([0., 0., 0., 1., 0., 0.]), 'theo': tensor([0., 0., 0., 0., 1., 0.]), 'yweweler': tensor([0., 0., 0., 0., 0., 1.])}


In [3]:
def normalization(x ,minimum, maximum):
    return (x - minimum) / (maximum - minimum + 1e-8)

def denormalization(x, minimum, maximum):
    return x * (maximum - minimum) + minimum

In [ ]:
path_spec = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\data_P3_P4\\P3\\spectrograms_padded"  
path = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P3"

In [5]:
def find_min_max(path_spec):
    min_val, max_val = float('inf'), float('-inf')
    count = 0

    for f in os.listdir(path_spec):
        if f.endswith('.npy'):
            data = np.load(os.path.join(path_spec, f), mmap_mode='r')  # Lecture sans tout charger en RAM
            count += 1
            min_val = min(min_val, np.min(data))
            max_val = max(max_val, np.max(data))

    return min_val, max_val, count

"""min_value, max_value, num = find_min_max(path_spec)
print(f"min_value : {min_value}")
print(f"max_value : {max_value}")
print(f"num : {num}")"""

min_value = -77.18305206298828
max_value = 46.953147888183594

In [ ]:
def create_mask():   
    path1 = path_spec
    path2 = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\data_P3_P4\\recordings"

    masks = {}
    max_len = 40

    for fichier in os.listdir(path2):
        if fichier.endswith(".wav"):
            path = os.path.join(path2, fichier)
            y, sr = librosa.load(path, sr=22050)

            # Calcul du spectrogramme
            stft = librosa.stft(y)
            spectrogram = np.abs(stft)
            log_spectrogram = librosa.amplitude_to_db(spectrogram)
            valid_len = log_spectrogram.shape[1] +2  # colonnes du vrai spectro 2 ppour la marge
            """valid_len = log_spectrogram.shape[1]   # colonnes du vrai spectro"""
            # Création du masque binaire
            mask_2d = np.zeros((513, 40), dtype=np.float32)
            mask_2d[:, :valid_len] = 1  # 1 sur les colonnes valides


            masks[fichier] = mask_2d  
    return masks  
masks = create_mask()

In [7]:
def list_spec_files(spec_path):
    return [os.path.join(spec_path, f) for f in os.listdir(spec_path) if f.endswith('.npy')]

def load_spec_normalized(file_path, minim, maxim):
    spec = np.load(file_path)
    spec_tensor = torch.from_numpy(spec).float()
    spec_norm = (spec_tensor - minim) / (maxim - minim)
    return spec_norm

def extract_speaker_name(file_path):
    filename = os.path.basename(file_path)
    return filename.split('_')[0] ,filename.split('_')[1]  # pour "0_nom_43.npy", donne "nom"

def load_dataset(spec_path, minim, maxim, masks):
    files = list_spec_files(spec_path)
    data = []
    for f in files:
        filename = os.path.basename(f).replace('.npy', '.wav')
        spec = load_spec_normalized(f, minim, maxim)
        nombre, speaker = extract_speaker_name(f)
        data.append((spec, masks[filename] ,nombre, speaker))
    return data


def get_data(spec_path, minim, maxim, masks, split_ratio=0.8, label = None):
    dataset =load_dataset(spec_path, minim, maxim, masks)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    train_dataset = DataLoader(train_dataset, batch_size=12, shuffle=True)
    test_dataset = DataLoader(test_dataset, batch_size=12, shuffle=False)
    return train_dataset, test_dataset

train_loader, test_loader = get_data(path_spec, min_value, max_value, masks)

In [8]:
def build_loss_vaea(lambda_reconstruct=0.5, lambda_kl=0.5):
    def loss_vae(x, x_hat, mean, logvar, mask=None):
        if mask is not None:
            mask = mask.view(mask.size(0), -1).to(x.device)
            
            recon_error = ((x - x_hat).pow(2)) * mask 
            valid = mask.sum()
            reconstruct_loss = lambda_reconstruct * (recon_error.sum() / (valid + 1e-8)) # on fait la moyenne plutot car mieux ici
        else:
            reconstruct_loss = lambda_reconstruct * (x - x_hat).pow(2).mean()

        KL_loss = -lambda_kl * torch.sum(1 + logvar - mean.pow(2) - logvar.exp()) / 2
        return reconstruct_loss + KL_loss

    return loss_vae

def build_loss_vae3(lambda_reconstruct=0.5, lambda_kl=0.5, lambda_l1=0.1):
    def loss_vae(x, x_hat, mean, logvar, mask=None):
        if mask is not None:
            mask = mask.view(mask.size(0), -1).to(x.device)
            recon_error = ((x - x_hat)**2) * mask
            valid = mask.sum()
            l2 = lambda_reconstruct * (recon_error.sum() / (valid + 1e-8))
            l1 = lambda_l1 * torch.abs(x - x_hat)[mask > 0].mean()
            reconstruct_loss = l2 + l1
        else:
            reconstruct_loss = lambda_reconstruct * (x - x_hat).pow(2).mean()

        KL_loss = -lambda_kl * torch.sum(1 + logvar - mean.pow(2) - logvar.exp()) / 2
        return reconstruct_loss + KL_loss

    return loss_vae






In [17]:
class CondVAEHybrid2(nn.Module):
    def __init__(self, in_channels=1, cond=True, num_classes=6, latent_dim=100):
        super(CondVAEHybrid2, self).__init__()
        self.cond = cond
        self.latent_dim = latent_dim
        self.num_classes = num_classes

        # Encoder: Reduced CNN depth, more fully-connected
        self.encoder_cnn = nn.Sequential(  # [B, 1, 1025, 90]
            nn.Conv2d(in_channels, 8, kernel_size=4, stride=2, padding=1),  # [B, 8, 512, 45]
            nn.ReLU(),
            nn.Conv2d(8, 16, kernel_size=4, stride=2, padding=1),           # [B, 16, 256, 23]
            nn.ReLU()
        )
        
        self.flatten_dim = 16*256*22
        self.fc1 = nn.Linear(self.flatten_dim + (num_classes if cond else 0), 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)

        self.fc_mean = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        # Decoder: Linear-heavy, lightweight CNN
        self.fc_decoder_input = nn.Sequential(
            nn.Linear(latent_dim + (num_classes if cond else 0), 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, self.flatten_dim),
            nn.ReLU()
        )

        self.decoder_cnn = nn.Sequential(
            nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1),  # [B, 8, ~512, ~46]
            nn.ReLU(),
            nn.ConvTranspose2d(8, 1, kernel_size=4, stride=2, padding=1),   # [B, 1, ~1024, ~90]
        )

    def encode(self, x, y=None):
        x = self.encoder_cnn(x)
        x = x.reshape(x.size(0), -1)
        if self.cond and y is not None:
            x = torch.cat([x, y], dim=1)
        
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc_mean(x), self.fc_logvar(x)

    def reparameterization(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mean + eps * std

    def decode(self, z, y=None):
        if self.cond and y is not None:
            z = torch.cat([z, y], dim=1)
        x = self.fc_decoder_input(z)
        x = x.view(-1, 16, 256, 22)
        x = self.decoder_cnn(x)
        x = F.interpolate(x, size=(1025, 90), mode='bilinear', align_corners=False)
        return x

    def forward(self, x, y=None):
        x = x.unsqueeze(1)
        mean, logvar = self.encode(x, y)
        z = self.reparameterization(mean, logvar)
        x_recon = self.decode(z, y)
        return x_recon, mean, logvar

    def transfer_speaker(self, x, speaker_from, speaker_to):
        with torch.no_grad():
            mean, logvar = self.encode(x, speaker_from)
            z = self.reparameterization(mean, logvar)
            out = self.decode(z, speaker_to)
            return out.squeeze(1).cpu().numpy()


In [19]:
def cond_train_model_vae(data_loader, model, criterion, optimizer, nepochs, scheduler = None, num_classes=6):
    #List to store loss to visualize


    for epoch in range(0, nepochs):
        train_loss = 0.
        model.train()
        for batch_idx, (input_,mask, nombre, personne) in enumerate(tqdm(data_loader, desc="Batch", leave=False)):         
            input_ = input_.to(device)
            
            y_one_hot = avoironehot(personne)
            y_one_hot = y_one_hot.to(device)
            optimizer.zero_grad()

            output, mean, logvar = model(input_,y_one_hot)
            
            if torch.isnan(output).any():
                print("NaN dans output")
            if torch.isnan(mean).any():
                print("NaN dans mean")
            if torch.isnan(logvar).any():
                print("NaN dans logvar")
            
            input_ = input_.view(input_.size(0),-1)
            loss = criterion(input_, output, mean, logvar, mask) #mettre mask
            if torch.isnan(loss).any():
                print("NaN dans loss")
            
            loss.backward()
            # Dans la boucle d'entraînement 
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            # update training loss
            train_loss += loss.item() * input_.size(0)
        if scheduler is not None:
            scheduler.step(train_loss)
        # calculate average losses
        train_loss = train_loss/len(data_loader.dataset)

        print('Epoch: {} \tTraining Loss: {:.6f}'.format(
            epoch, train_loss))
        

In [20]:
# model
input_dim = 513 * 40  # = 92250
cond_dim = 6  # nombre de classes

model = CondVAEHybrid2().to(device)

# loss function
criterion = build_loss_vaea()

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)


nepochs = 10

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, verbose=True)

cond_train_model_vae(train_loader,model, criterion, optimizer, nepochs,  scheduler= scheduler)

PATH = "model2.pth"

# Sauvegarder les paramètres du modèle
torch.save(model.state_dict(), PATH)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (12x20486 and 90118x1024)

In [180]:
def save_audio_original(path, spec, mask= None):

    if mask is not None:
        spec = np.where(mask > 0, spec, -80)
    audio_path = os.path.join(path, f"audio_original.wav")
    spec_amp = librosa.db_to_amplitude(spec)
    signal = librosa.griffinlim(spec_amp, hop_length=512, n_fft=1024)
    signal = librosa.effects.time_stretch(signal, rate=0.40)

    sf.write(audio_path, signal, 22050)


    
def save_audio_reconstructed(path, spec, mask= None):
    
    # Enregistrer le fichier audio
    if mask is not None:
        spec = np.where(mask > 0, spec, -80)
    audio_path = os.path.join(path , f"audio_reconstruit.wav")
    spec_amp = librosa.db_to_amplitude(spec)
    signal = librosa.griffinlim(spec_amp, hop_length=512, n_fft=1024)

    # Post-traitement : ralentir de 5%
    signal = librosa.effects.time_stretch(signal, rate=0.40)
    

    sf.write(audio_path, signal, 22050)

def save_image_original(path, image, mask= None):
    
    if mask is not None:
        image = np.where(mask > 0, image, -80)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(image,sr=22050,hop_length=512, n_fft=1024,
            y_axis='log',x_axis='time',vmin=-80, vmax=60,cmap='magma'  
        )
    plt.colorbar(format='%+2.0f dB')
    plt.title(f"Original spectrogram :")
    img_path = os.path.join(path, f"image_original.png")
    plt.savefig(img_path)
    plt.close()



def save_image_reconstruit(path, image, mask= None):
    
    if mask is not None:
        image = np.where(mask > 0, image, -80)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(image,sr=22050,hop_length=512, n_fft=1024,
            y_axis='log',x_axis='time',vmin=-80, vmax=60,cmap='magma'  
        )
    plt.colorbar(format='%+2.0f dB')
    plt.title(f"reconstruit spectrogram :")
    img_path = os.path.join(path, f"image_reconstruite.png")
    plt.savefig(img_path)
    plt.close()

In [ ]:
def changement_de_personne(path_spec, a_name, b_name):
    a_onehot = avoironehot(a_name)
    b_onehot = avoironehot(b_name)

    for fichier in os.listdir(path_spec):
        if fichier.endswith('.npy'):
            
            file_path = os.path.join(path_spec, fichier)

            nom = fichier.split("_")[-2]
            if nom == a_name[0]:
                data = np.load(file_path)
                print("Fichier :", file_path)
                spec = torch.from_numpy(data).float()
                spec = (spec - min_value) / (max_value - min_value)
                x = spec.unsqueeze(0).to(device).contiguous()  # ← ajoute .contiguous() [1, 1025, 40]

                print(a_onehot)
                print(b_onehot)

                changement = model.transfer_speaker(x, a_onehot.to(device), b_onehot.to(device))

                print("Fichier :", file_path)
                print("Ancien :", a_name)
                print("Nouveau :", b_name)
                print("Output shape :", changement.shape)
                mask = masks[fichier.replace('.npy', '.wav')]
                return x , changement, a_name, b_name, mask

                
x , changement, a_name, b_name, mask = changement_de_personne(path_spec, ["lucas"], ["george"])
x = denormalization(x, min_value, max_value)
changement = denormalization(changement, min_value, max_value)
path_save = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P3\\test"
print(x.squeeze().shape)
print(changement.shape)
x= x.squeeze().cpu().numpy()
path_changement = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P3\\changement_conv"
save_image_original(path_changement, x, mask)
save_image_reconstruit(path_changement, changement, mask)
save_audio_original(path_changement, x, mask)
save_audio_reconstructed(path_changement, changement, mask)





Fichier : c:\Users\gabri\Desktop\dauphine\L3\S2\Deep_learning\nouveau_projet\spectrograms_padded\0_lucas_0.npy
tensor([[0., 0., 1., 0., 0., 0.]])
tensor([[1., 0., 0., 0., 0., 0.]])
Fichier : c:\Users\gabri\Desktop\dauphine\L3\S2\Deep_learning\nouveau_projet\spectrograms_padded\0_lucas_0.npy
Ancien : ['lucas']
Nouveau : ['george']
Output shape : (513, 40)
torch.Size([513, 40])
(513, 40)


In [182]:
def reconstruire_chaque_personne():
    # Get list of unique speakers
    speakers = ['george', 'jackson', 'lucas', 'nicolas', 'theo', 'yweweler']
    
    # Create output directory if it doesn't exist
    output_dir = "c:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\nouveau_projet\\final\\reconstructions"
    os.makedirs(output_dir, exist_ok=True)
    
    # For each speaker
    for speaker in speakers:
        # Create speaker directory
        speaker_dir = os.path.join(output_dir, speaker)
        os.makedirs(speaker_dir, exist_ok=True)
        noms = []
        # Find all files for this speaker
        for fichier in os.listdir(path_spec):
            if fichier.endswith('.npy'):
                file_path = os.path.join(path_spec, fichier)
                nom = fichier.split("_")[-2]
                
                if nom == speaker and nom not in noms:
                    noms.append(speaker)
                    # Load and normalize spectrogram
                    data = np.load(file_path)
                    spec = torch.from_numpy(data).float()
                    spec = (spec - min_value) / (max_value - min_value)
                    x = spec.unsqueeze(0).to(device).contiguous()
                    
                    # Get speaker one-hot encoding
                    speaker_onehot = avoironehot([speaker]).to(device)
                    
                    # Reconstruct using the model
                    with torch.no_grad():
                        reconstructed = model.reconstruct(x, speaker_onehot)
                        reconstructed = reconstructed.view(513 , 40)
                        reconstructed = reconstructed.squeeze().cpu().numpy()
                    
                    # Denormalize
                    x = denormalization(x, min_value, max_value)
                    reconstructed = denormalization(reconstructed, min_value, max_value)
                    
                    # Save original and reconstructed spectrograms
                    mask = masks[fichier.replace('.npy', '.wav')]
                    save_image_original(speaker_dir, x.squeeze().cpu().numpy(),mask)
                    save_image_reconstruit(speaker_dir, reconstructed,mask)
                    
                    # Save original and reconstructed audio
                    save_audio_original(speaker_dir, x.squeeze().cpu().numpy(),mask)
                    save_audio_reconstructed(speaker_dir, reconstructed,mask)
                    
                    print(f"Processed {fichier} for speaker {speaker}")
                    
reconstruire_chaque_personne()

Processed 0_george_0.npy for speaker george
Processed 0_jackson_0.npy for speaker jackson
Processed 0_lucas_0.npy for speaker lucas
Processed 0_nicolas_0.npy for speaker nicolas
Processed 0_theo_0.npy for speaker theo
Processed 0_yweweler_0.npy for speaker yweweler
